# Ранжирование документов: AMML Ranking Competition 2026

Двухэтапный реранкинг с дообучением Jina Reranker v3.5 (LoRA), повторным ранжированием и калибровкой. Метрика — NDCG@10. Среда: Linux, Kaggle, GPU T4 ×2; входные файлы — конкурсные train и test. Итоговый файл — `submission.csv`.

## Окружение и параметры

In [1]:
import os, sys, subprocess, platform, time
os.environ["V13_FINAL_START"] = str(time.monotonic())

TOTAL_MINUTES = 290
os.environ["V13_TOTAL_MINUTES"] = str(TOTAL_MINUTES)
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError

if platform.system() != 'Linux':
    raise RuntimeError('Этот ноутбук подготовлен для Kaggle Linux / GPU T4 ×2.')
if Path('/kaggle/working').is_dir():
    os.chdir('/kaggle/working')
os.environ.update(USE_TF='0', USE_FLAX='0', TOKENIZERS_PARALLELISM='false')
try:
    version('torch')
except PackageNotFoundError:
    raise RuntimeError('Выберите GPU-образ Kaggle с CUDA PyTorch.')
pins = {
    'packaging': '25.0',
    'numpy': '2.2.6' if sys.version_info >= (3, 13) else '1.26.4',
    'pandas': '2.2.3', 'transformers': '4.57.3', 'peft': '0.18.0',
    'accelerate': '1.6.0', 'huggingface-hub': '0.36.0',
    'tokenizers': '0.22.1', 'safetensors': '0.5.3', 'tqdm': '4.67.1',
    'snowballstemmer': '2.2.0',
}
missing = []
for package, expected in pins.items():
    try:
        installed = version(package)
    except PackageNotFoundError:
        installed = None
    if installed != expected:
        missing.append(f'{package}=={expected}')
if missing:
    command = [sys.executable, '-m', 'pip', 'install',
               '--disable-pip-version-check', '--only-binary=:all:', *missing]
    with Path('pip_jina.log').open('w') as log:
        status = subprocess.run(command, stdout=log, stderr=subprocess.STDOUT, timeout=900).returncode
    if status:
        detail = Path('pip_jina.log').read_text(errors='replace')[-14000:]
        raise RuntimeError('Ошибка установки. Проверьте Internet → On.\n' + detail)

In [2]:

RUN_MODE = 'validate_and_submit'
TRAIN_FILE = ''
TEST_FILE = ''
for key,value in [('AMML_RUN_MODE',RUN_MODE),('AMML_TRAIN_FILE',TRAIN_FILE),('AMML_TEST_FILE',TEST_FILE)]:
    if value: os.environ[key]=value
    else: os.environ.pop(key,None)

## Подготовка данных, обучение и ранжирование

In [3]:
%%writefile _jina_v13_extended.py
import sys, time, pickle, subprocess, argparse
import os, gc, json, math, re, html, hashlib, unicodedata, importlib, threading, inspect, platform
from dataclasses import dataclass, asdict, replace
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

os.environ['USE_TF'] = '0'
os.environ['USE_FLAX'] = '0'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ.setdefault('HF_HOME', str((Path('/kaggle/temp') if Path('/kaggle/temp').is_dir() else Path('/tmp')) / 'amml_hf_cache'))
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm
import transformers
import transformers.utils.import_utils as optional_imports

optional_imports._torchvision_available = False
optional_imports._librosa_available = False
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup
from transformers.dynamic_module_utils import get_class_from_dynamic_module
from peft import LoraConfig, TaskType, get_peft_model
import peft
import peft.tuners.lora.torchao as optional_quantizers
optional_quantizers.is_torchao_available = lambda: False
from contextlib import nullcontext

PIPELINE_VERSION = 'v13-safe-extended-290min'
TOTAL_MINUTES = int(os.getenv('V13_TOTAL_MINUTES', '290')) 

@dataclass(frozen=True)
class Config:
    model: str = 'jinaai/jina-reranker-v3.5'
    revision: str = 'e8a93f33f0b22108f8c2364f8484ce3422552fbc'
    seeds: tuple = (2026, 2027)
    planned_epochs: int = 1
    stop_epochs: int = 1
    max_length: int = 4096
    doc_tokens: int = 448
    query_tokens: int = 96
    title_tokens: int = 48
    rank: int = 8
    lr: float = 1e-5
    projector_lr: float = 5e-6
    accum: int = 8
    temperature: float = 0.1
    checkpoint_every: int = 40

CFG = Config()
SPECIAL = {'doc_embed_token': '<|embed_token|>', 'query_embed_token': '<|rerank_token|>'}
LOAD_LOCK = threading.Lock()


MOJIBAKE = re.compile(r'(?:Р[\u0400-\u052f]|С[\u0400-\u052f])')
MIXED_MARKS = re.compile(r'[\u0301\u0341\u200b\u200c\u200d\ufeff\u00ad]')
CONTROL = re.compile(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]')

def repair_encoding(s):

    bad=len(MOJIBAKE.findall(s))
    if bad >= max(2, len(s)//25):
        try:
            candidate=s.encode('cp1251').decode('utf-8')
            if len(MOJIBAKE.findall(candidate)) <= bad//4 and len(candidate)<len(s):
                return candidate
        except (UnicodeEncodeError,UnicodeDecodeError):
            pass
    return s

def clean_text(value):
    if pd.isna(value): return ''
    text=repair_encoding(str(value))

    text=unicodedata.normalize('NFC', unicodedata.normalize('NFKC',html.unescape(text)))
    text=MIXED_MARKS.sub('',text)
    text=CONTROL.sub(' ',text)
    text=re.sub(r'</?(?:br|p|div|span|b|i|strong|em|a|sup|sub|li|ref)\b[^>]*>', ' ',text,flags=re.I)
    return re.sub(r'\s+', ' ',text).strip()


import signal, shutil
from torch import nn
BRANCHES = {"full":dict(seed=2026,last_layers=16,focus=False,residual=False), "short":dict(seed=2027,last_layers=8,focus=True,residual=False)}
SHORT_K=6
SHORT_K_HARD=8
SHORT_WEIGHT=.45
CHUNK_SIZE=128
HARD_FRACTION=.40

def find_data_files():
    required = {'id', 'query_id', 'document_id', 'article_id', 'query', 'title', 'text'}
    candidates = {'train': [], 'test': []}
    roots = [Path('/kaggle/input'), Path(os.getenv('AMML_DATA_DIR', 'data'))]
    expected = {'train': 31530, 'test': 13507}
    for root in roots:
        if not root.is_dir():
            continue
        for path in root.rglob('*'):
            if not path.is_file() or not path.name.lower().endswith(('.csv', '.csv.gz')):
                continue
            try:
                columns = set(pd.read_csv(path, nrows=0, encoding='utf-8-sig').columns)
                if required <= columns:
                    role = 'train' if 'relevance' in columns else 'test'
                    count = len(pd.read_csv(path, usecols=['id']))
                    candidates[role].append((count, str(path.resolve())))
            except (OSError, ValueError, UnicodeError, pd.errors.ParserError):
                continue
    result = {}
    for role in ['train', 'test']:
        override = os.getenv(f'AMML_{role.upper()}_FILE')
        if override:
            result[role] = Path(override).expanduser()
        elif candidates[role]:
            exact = [x for x in candidates[role] if x[0] == expected[role]]
            if not exact:
                raise ValueError(f'Не найден полный {role}: {expected[role]} строк. Кандидаты: {candidates[role]}')
            result[role] = Path(sorted(set(exact))[0][1])
        else:
            raise FileNotFoundError(f'Не найден {role}. Подключите конкурсные CSV через Add Input.')
    return result


def read_data(paths):
    options = dict(encoding='utf-8-sig', low_memory=False,
                   dtype={key: str for key in ['id', 'query_id', 'document_id', 'article_id']})
    train = pd.read_csv(paths['train'], **options)
    test = pd.read_csv(paths['test'], **options)
    if 'relevance' not in train or 'relevance' in test:
        raise ValueError('Неверные роли train/test.')
    if not train.id.is_unique or not test.id.is_unique or set(train.id) & set(test.id):
        raise ValueError('Идентификаторы строк должны быть уникальными.')
    if not set(train.relevance.unique()) <= {0, 1}:
        raise ValueError('Ожидались бинарные relevance.')
    if (len(train), len(test), train.query_id.nunique(), test.query_id.nunique()) != (31530, 13507, 3042, 1305):
        raise ValueError('Размеры конкурсных данных не совпадают с условиями.')
    return train.reset_index(drop=True), test.reset_index(drop=True)


def clean_data(raw_train, raw_test):
    whole = pd.concat([raw_train.drop(columns='relevance'), raw_test], ignore_index=True)
    for key in ['id', 'query_id', 'document_id', 'article_id']:
        if whole[key].isna().any() or whole[key].eq('').any():
            raise ValueError(f'Пропуски в ключе {key}.')
        whole[key] = whole[key].astype(str)
    for column in ['query', 'title', 'text']:
        whole[column] = whole[column].map(clean_text)


    for key in ['document_id', 'article_id']:
        known = whole.loc[whole.title.ne(''), [key,'title']].drop_duplicates()
        counts = known.groupby(key).title.nunique()
        lookup = known[known[key].isin(counts[counts==1].index)].set_index(key).title
        missing = whole.title.eq('')
        whole.loc[missing,'title'] = whole.loc[missing,key].map(lookup).fillna('')

    def canonical(values):
        values = values[values.ne('')]
        if values.empty:
            raise ValueError('Все тексты одного запроса пусты.')
        counts = values.value_counts()
        return sorted(counts[counts == counts.max()].index)[0]
    queries = whole.groupby('query_id')['query'].agg(canonical)
    whole['query'] = whole.query_id.map(queries)
    train = whole.iloc[:len(raw_train)].reset_index(drop=True).copy()
    test = whole.iloc[len(raw_train):].reset_index(drop=True).copy()
    train['relevance'] = raw_train.relevance.to_numpy(dtype='int8')
    if set(train.query_id) & set(test.query_id):
        raise ValueError('Один query_id находится одновременно в train и test.')
    return train, test


def ndcg_vector(frame, prediction):
    prediction = np.asarray(prediction, dtype=float)
    if len(prediction) != len(frame) or not np.isfinite(prediction).all():
        raise ValueError('Невалидный вектор prediction.')
    result = []
    for positions in frame.groupby('query_id', sort=True).indices.values():
        group = frame.iloc[positions]
        labels = group.relevance.to_numpy(dtype=int)
        order = np.lexsort((group.id.astype(str).to_numpy(), -prediction[positions]))
        top = labels[order[:10]]
        dcg = np.sum(top / np.log2(np.arange(2, len(top) + 2)))
        count = min(10, int(labels.sum()))
        ideal = np.sum(1.0 / np.log2(np.arange(2, count + 2)))
        result.append(float(dcg / ideal) if ideal else 0.0)
    return np.asarray(result)


def normalize_by_query(frame, prediction):
    result = np.empty(len(frame), dtype=float)
    for positions in frame.groupby('query_id', sort=False).indices.values():
        scores = np.asarray(prediction)[positions]
        result[positions] = (scores - scores.mean()) / max(float(scores.std()), 1e-8)
    return result


def numeric_values(s):
    text=s.astype('string').str.replace(r'[\s\u00a0\u202f]', '',regex=True).str.replace(',','.',regex=False)
    return pd.to_numeric(text,errors='coerce')


def audit_data(raw_train,raw_test,train,test,root):
    reports=[]; sources=[]
    for name,raw,cleaned in [('train',raw_train,train),('test',raw_test,test)]:
        lengths=raw.text.fillna('').astype(str).str.len(); sizes=raw.groupby('query_id').size()
        row=dict(dataset=name,rows=len(raw),queries=len(sizes),title_missing=int(raw.title.fillna('').str.strip().eq('').sum()),
                 title_empty_after=int(cleaned.title.eq('').sum()),recovered_titles=int((raw.title.fillna('').str.strip().eq('')&cleaned.title.ne('')).sum()),
                 text_chars_p50=float(lengths.median()),text_chars_p99=float(lengths.quantile(.99)),max_candidates=int(sizes.max()),
                 duplicate_documents=int(raw.document_id.duplicated().sum()),duplicate_rows=int(raw.id.duplicated().sum()),
                 accent_rows=int(raw['query'].str.contains('\u0301',regex=False,na=False).sum()))
        for field in ['passage_index','article_passage_count','article_char_count']:
            num=numeric_values(raw[field]); row[field+'_missing_or_invalid']=int(num.isna().sum()); row[field+'_negative']=int(num.lt(0).sum())
        reports.append(row)
        for source,sub in raw.groupby('ingest_source',dropna=False):
            sources.append(dict(dataset=name,source=str(source),rows=len(sub),title_missing=int(sub.title.isna().sum()),
                                char_count_missing_or_invalid=int(numeric_values(sub.article_char_count).isna().sum()),
                                median_chars=float(sub.text.fillna('').str.len().median())))
    pd.DataFrame(reports).to_csv(root/'data_audit.csv',index=False)
    pd.DataFrame(sources).to_csv(root/'source_audit.csv',index=False)
    g=train.groupby('query_id').relevance.agg(['size','sum'])
    mixed=train.groupby(['query_id','article_id']).relevance.nunique().gt(1).groupby(level=0).any()
    stats=dict(train_single_positive=int(g['sum'].eq(1).sum()),mixed_article_queries=int(mixed.sum()),
               document_overlap=len(set(train.document_id)&set(test.document_id)),article_overlap=len(set(train.article_id)&set(test.article_id)))
    atomic_json(root/'audit_summary.json',stats)
    if train.query_id.nunique()!=raw_train.query_id.nunique() or len(train)!=len(raw_train) or len(test)!=len(raw_test): raise AssertionError('Изменён состав кандидатов')


def connected_query_split(frame, seed=839127):
    parent = {q: q for q in frame.query_id.unique()}
    def find(q):
        while parent[q] != q:
            parent[q] = parent[parent[q]]
            q = parent[q]
        return q
    def union(a, b):
        a, b = find(a), find(b)
        if a != b:
            parent[a] = b
    for _, group in frame.groupby('document_id', sort=True):
        queries = group.query_id.unique()
        for query in queries[1:]:
            union(queries[0], query)
    normalized = frame['query'].str.casefold().str.replace('ё', 'е', regex=False)
    for _, group in frame.groupby(normalized, sort=True):
        queries = group.query_id.unique()
        for query in queries[1:]:
            union(queries[0], query)
    component = frame.query_id.map(find)
    unique = np.sort(component.unique())
    if len(unique) < 10:
        raise ValueError('Недостаточно независимых групп для валидации.')
    shuffled = np.random.default_rng(seed).permutation(unique)
    held_components = set(shuffled[:math.ceil(0.20 * len(unique))])
    mask = component.isin(held_components).to_numpy()
    fit, held = np.flatnonzero(~mask), np.flatnonzero(mask)
    left, right = frame.iloc[fit], frame.iloc[held]
    for key in ['query_id', 'document_id']:
        if not set(left[key]).isdisjoint(right[key]):
            raise AssertionError(f'Утечка между fit/validation по {key}.')
    if not set(normalized.iloc[fit]).isdisjoint(normalized.iloc[held]):
        raise AssertionError('Одинаковый текст запроса попал в разные части.')
    if right.query_id.nunique() < 100:
        raise ValueError('Для проверки нужно не менее 100 отложенных запросов.')
    return fit, held, {q: find(q) for q in parent}


def atomic_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding='utf-8')
    temporary.replace(path)


def read_json(path):
    try:
        return json.loads(Path(path).read_text(encoding='utf-8'))
    except (OSError, ValueError):
        return {}


def save_array(path,values):
    path=Path(path); temp=path.with_suffix('.tmp.npy'); np.save(temp,values,allow_pickle=False); temp.replace(path)


SUBMISSION_ORDER_CACHE={}

def write_submission(raw_test,scores,path):
    scores=np.asarray(scores,dtype=float)
    if scores.shape!=(len(raw_test),) or not np.isfinite(scores).all(): raise ValueError('Invalid scores')
    sub=pd.DataFrame({'id':raw_test.id.astype(str),'prediction':scores})
    if not sub.id.is_unique: raise ValueError('Duplicate ids')

    key=(len(sub),str(sub.id.iloc[0]),str(sub.id.iloc[-1]))
    if key not in SUBMISSION_ORDER_CACHE:
        sample_order=None
        for p in Path('/kaggle/input').rglob('*.csv'):
            try:
                cols=pd.read_csv(p,nrows=0).columns.tolist()
                if set(cols)=={'id','prediction'}:
                    sample=pd.read_csv(p,usecols=['id'],dtype={'id':str})
                    if len(sample)==len(sub) and sample.id.is_unique and set(sample.id)==set(sub.id):
                        sample_order=sample; break
            except (OSError,ValueError): continue
        SUBMISSION_ORDER_CACHE[key]=sample_order
    sample=SUBMISSION_ORDER_CACHE[key]
    if sample is not None: sub=sample.merge(sub,on='id',validate='one_to_one',how='left')
    path=Path(path); temp=path.with_suffix('.tmp.csv'); sub.to_csv(temp,index=False,float_format='%.17g'); temp.replace(path)
    check=pd.read_csv(path,dtype={'id':str}); assert check.columns.tolist()==['id','prediction'] and set(check.id)==set(raw_test.id.astype(str)) and check.id.is_unique and np.isfinite(check.prediction).all()
    return len(check)


def export_lock(path):
    from importlib.metadata import version, requires
    from packaging.requirements import Requirement
    from packaging.utils import canonicalize_name
    queue=['torch','numpy','pandas','transformers','peft','accelerate','huggingface-hub','tokenizers','safetensors','snowballstemmer','tqdm','packaging','ipykernel','ipython','nbformat']; packages={}
    while queue:
        name=canonicalize_name(queue.pop())
        if name in packages: continue
        packages[name]=version(name)
        for item in requires(name) or []:
            dep=Requirement(item)
            if dep.marker is None or dep.marker.evaluate({'extra':''}): queue.append(dep.name)
    lines=[f'# Linux / Python {platform.python_version()} / CUDA {torch.version.cuda}']
    flavor=packages['torch'].partition('+')[2]
    if re.fullmatch(r'cu\d+|cpu',flavor): lines.append(f'--extra-index-url https://download.pytorch.org/whl/{flavor}')
    lines.extend(f'{name}=={ver}' for name,ver in sorted(packages.items())); Path(path).write_text('\n'.join(lines)+'\n',encoding='utf-8')


from functools import lru_cache
from snowballstemmer import stemmer
STEMMERS = threading.local()
STOP = set('и в во на у к с со из по о об от до для при под над за а но или не что кто где когда как какой какая какие какое который которая которых ли был была были есть это сколько чем почему'.split())
WORD = re.compile(r'[а-яёa-z0-9]+', re.I)

@lru_cache(maxsize=100000)
def stem_word(word):
    if not hasattr(STEMMERS, 'russian'):
        STEMMERS.russian = stemmer('russian')
    return STEMMERS.russian.stemWord(word.casefold().replace('ё', 'е'))


def query_terms(query):
    return {stem_word(w) for w in WORD.findall(clean_text(query))
            if len(w) > 2 and w.casefold() not in STOP}


def focused_excerpt(ids, hits, budget):
    if len(ids) <= budget:
        return ids
    if not hits:
        return head_tail(ids, budget)

    starts = {0, max(0, len(ids)-budget)}
    for positions in hits.values():
        for pos in positions:
            starts.add(max(0, min(len(ids)-budget, pos-budget//3)))
            starts.add(max(0, min(len(ids)-budget, pos-budget+1)))
    def quality(start):
        counts = [sum(start <= p < start+budget for p in positions) for positions in hits.values()]
        return (sum(c > 0 for c in counts), sum(min(c, 2) for c in counts), -start)
    start = max(starts, key=quality)
    return ids[start:start+budget]


def fair_budgets(lengths, total, floor=24):
    sizes=np.asarray(lengths,dtype=int)
    if total<0 or np.any(sizes<0): raise ValueError('Negative token budget')
    if sizes.sum()<=total: return sizes.copy()
    low,high=0,int(sizes.max())
    while low<high:
        mid=(low+high+1)//2
        if np.minimum(sizes,mid).sum()<=total: low=mid
        else: high=mid-1
    result=np.minimum(sizes,low)
    left=total-int(result.sum())
    for i in np.flatnonzero(result<sizes)[:left]: result[i]+=1
    if np.any(result<np.minimum(sizes,floor)):
        raise ValueError('Too little context for every candidate')
    return result


def prepare_groups(frame, tokenizer, cfg, focus):
    groups=[]
    passage=numeric_values(frame.passage_index).fillna(-1).to_numpy()
    for q,positions in tqdm(frame.groupby('query_id',sort=False).indices.items(),desc='Tokenize lists',mininterval=15):
        positions=np.asarray(positions); positions=positions[np.argsort(frame.iloc[positions].id.to_numpy(),kind='stable')]
        rows=frame.iloc[positions]; query=tokenizer.decode(text_ids(rows['query'].iloc[0],tokenizer,cfg.query_tokens))
        docs=[]; prefixes=[]; hits=[]; terms=query_terms(query) if focus else set()
        for row in rows.itertuples():
            title=tokenizer.decode(text_ids(row.title,tokenizer,cfg.title_tokens))
            prefix=tokenizer.encode(f'Заголовок статьи: {title}\nФрагмент: ',add_special_tokens=False)
            text=sanitize(str(row.text),tokenizer)
            if focus and terms:
                enc=tokenizer(text,add_special_tokens=False,return_offsets_mapping=True)
                ids=enc['input_ids']; ends=np.asarray([end for _,end in enc['offset_mapping']]); h={}
                if len(ends):
                    for match in WORD.finditer(text):
                        term=stem_word(match.group())
                        if term in terms:
                            i=min(len(ends)-1,int(np.searchsorted(ends,match.start(),side='right')))
                            h.setdefault(term,[]).append(i)
            else:
                ids=tokenizer.encode(text,add_special_tokens=False); h={}
            docs.append(ids); prefixes.append(prefix); hits.append(h)
        _,arts=np.unique(rows.article_id.to_numpy(str),return_inverse=True)
        groups.append(dict(query_id=str(q),positions=positions,query=query,docs=docs,prefixes=prefixes,hits=hits,articles=arts,
            passage=passage[positions],labels=rows.relevance.to_numpy(dtype=np.float32) if 'relevance' in rows else None))
    return groups


def candidate_order(group, seed, training=False, rng=None, reverse=False):

    if rng is None:
        local=int.from_bytes(hashlib.blake2b(group['query_id'].encode(),digest_size=8).digest(),'little')
        rng=np.random.default_rng(local ^ seed)
    blocks=[]
    for a in np.unique(group['articles']):
        ix=np.flatnonzero(group['articles']==a)
        ix=ix[np.argsort(group['passage'][ix],kind='stable')]
        if training and rng.random()<0.5: ix=ix[::-1]
        blocks.append(ix)
    order=np.concatenate([blocks[i] for i in rng.permutation(len(blocks))])
    return order[::-1].copy() if reverse else order.copy()


def pack_group(group, tokenizer, formatter, cfg, order, focus):
    count=len(group['docs']); order=np.asarray(order,dtype=int)
    if sorted(order.tolist())!=list(range(count)): raise ValueError('Invalid document order')
    prefixes=[group['prefixes'][i] for i in order]

    lengths=[min(len(group['docs'][i]),max(24,cfg.doc_tokens-len(p))) for i,p in zip(order,prefixes)]
    budget=sum(lengths)
    for attempt in range(30):
        canonical=np.argsort(order); limits=np.empty(count,dtype=int)
        limits[canonical]=fair_budgets(np.asarray(lengths)[canonical],budget)
        docs=[]
        for i,p,k in zip(order,prefixes,limits):
            ids=group['docs'][i]; k=int(k)
            excerpt=focused_excerpt(ids,group['hits'][i],k) if focus else head_tail(ids,k)
            docs.append(tokenizer.decode(p+excerpt))
        prompt=formatter(group['query'],docs,special_tokens=SPECIAL,no_thinking=True)
        encoded=tokenizer(prompt,add_special_tokens=False,return_tensors='pt'); ids=encoded.input_ids
        if ids.shape[1]<=cfg.max_length:
            if int((ids==151670).sum())!=count or int((ids==151671).sum())!=1: raise ValueError('Jina markers missing')
            kept=np.empty(count,dtype=int); kept[order]=limits
            return dict(encoded),kept
        budget-=max(8,ids.shape[1]-cfg.max_length+count)
    raise RuntimeError('Cannot pack group without dropping candidates')

def load_official_tools(cfg):
    cls = get_class_from_dynamic_module(
        'modeling.JinaForRanking', cfg.model,
        revision=cfg.revision, code_revision=cfg.revision,
    )
    formatter = importlib.import_module(cls.__module__).format_docs_prompts_func
    tokenizer = AutoTokenizer.from_pretrained(cfg.model, revision=cfg.revision)
    expected = [151670, 151671]
    actual = tokenizer.convert_tokens_to_ids(list(SPECIAL.values()))
    if actual != expected:
        raise ValueError(f'Изменились специальные токены Jina: {actual}')
    tokenizer.padding_side = 'left'
    return tokenizer, formatter


def sanitize(text, tokenizer):
    for marker in tokenizer.all_special_tokens:
        text = text.replace(marker, '')
    return text


def head_tail(ids, limit):
    if len(ids) <= limit:
        return ids
    head = max(1, 3 * limit // 4)
    return ids[:head] + ids[-(limit - head):] if limit > head else ids[:limit]


def text_ids(text, tokenizer, limit):
    return head_tail(tokenizer.encode(sanitize(str(text), tokenizer), add_special_tokens=False), limit)


def ranking_loss(scores, labels, temperature=0.1):
    scores = scores.float() / temperature
    labels = labels.float()
    positive = torch.nonzero(labels > 0, as_tuple=True)[0]
    negative = torch.nonzero(labels == 0, as_tuple=True)[0]
    if not len(positive) or not len(negative):
        return scores.sum() * 0

    listwise = -F.log_softmax(scores, dim=0)[positive].mean()

    with torch.no_grad():
        order = torch.argsort(scores, descending=True, stable=True)
        ranks = torch.empty_like(order)
        ranks[order] = torch.arange(len(order), device=scores.device)
        discounts = 1.0 / torch.log2(ranks.float() + 2)
        discounts[ranks >= 10] = 0
        ideal = (1.0 / torch.log2(torch.arange(
            2, min(10, len(positive)) + 2, device=scores.device, dtype=torch.float32))).sum()
        weights = (discounts[positive, None] - discounts[negative][None, :]).abs() / ideal
    differences = scores[positive, None] - scores[negative][None, :]
    pairwise = (weights * F.softplus(-differences)).sum() / weights.sum().clamp_min(1e-8)
    return 0.7 * pairwise + 0.3 * listwise


def make_model(cfg, spec, device):
    torch.manual_seed(spec['seed']); torch.cuda.manual_seed_all(spec['seed'])
    model=AutoModel.from_pretrained(cfg.model,revision=cfg.revision,code_revision=cfg.revision,trust_remote_code=True,
       torch_dtype=torch.float16,attn_implementation='sdpa',low_cpu_mem_usage=True).to(device)
    model.config.use_cache=False; model.lm_head=nn.Identity()
    suffix={'q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'}
    candidates=[]
    for name,module in model.named_modules():
        match=re.search(r'(?:^|\.)layers\.(\d+)\.',name)
        if match and name.split('.')[-1] in suffix and isinstance(module,nn.Linear): candidates.append((name,int(match.group(1))))
    if not candidates: raise RuntimeError('Cannot locate transformer projections by layer; refusing silent fallback')
    layer_ids=sorted({i for _,i in candidates}); keep=set(layer_ids[-spec['last_layers']:]); targets=[n for n,i in candidates if i in keep]
    if len(keep)!=spec['last_layers']: raise RuntimeError(f'Unexpected layer count {len(layer_ids)}')
    lora=LoraConfig(task_type=TaskType.FEATURE_EXTRACTION,r=cfg.rank,lora_alpha=2*cfg.rank,lora_dropout=.05,
         target_modules=targets,modules_to_save=['projector'])
    model=get_peft_model(model,lora)
    for param in model.parameters():
        if param.requires_grad: param.data=param.data.float()


    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant':False})
    tokenizer,formatter=load_official_tools(cfg)
    return model,tokenizer,formatter


def embeddings(model,batch,device):
    batch={k:v.to(device) for k,v in batch.items()}
    with torch.autocast('cuda',dtype=torch.float16,enabled=device.type=='cuda'):
        output=model(**batch)
    docs=output.doc_embeds.float()[0]; query=output.query_embeds.float().reshape(-1,docs.shape[-1])
    if docs.ndim!=2 or query.shape[0]!=1: raise RuntimeError(f'Unexpected Jina embeddings {docs.shape} {query.shape}')
    return docs,query


def forward_scores(model,head,batch,articles,device):
    docs,query=embeddings(model,batch,device)
    base=F.cosine_similarity(docs,query.expand_as(docs),dim=-1)
    score=base if head is None else base+head(docs,query,articles,base)
    return score,base


def make_optimizer(model,head,cfg,updates):
    proj=[]; lora=[]
    for name,p in model.named_parameters():
        if p.requires_grad: (proj if 'projector' in name else lora).append(p)
    if not proj or not lora: raise RuntimeError('Missing trainable projector or LoRA')
    groups=[{'params':lora,'lr':cfg.lr},{'params':proj,'lr':cfg.projector_lr}]
    if head is not None: groups.append({'params':list(head.parameters()),'lr':3e-4})
    optimizer=torch.optim.AdamW(groups,weight_decay=.01)

    warm=max(2,int(.04*updates))
    scheduler=torch.optim.lr_scheduler.LambdaLR(optimizer,lambda k:min(1.,(k+1)/warm)*max(.35,1-.65*k/max(1,updates)))
    return optimizer,scheduler,[p for g in groups for p in g['params']]

def save_checkpoint(model,head,root,progress):
    from peft import get_peft_model_state_dict
    state={'adapter':{k:v.detach().cpu() for k,v in get_peft_model_state_dict(model).items()},
           'head':None if head is None else {k:v.detach().cpu() for k,v in head.state_dict().items()},'progress':progress}
    tmp=root/'checkpoint.tmp.pt'; torch.save(state,tmp); tmp.replace(root/'checkpoint.pt')


def prepare_one(group,tokenizer,formatter,cfg,spec,order):
    batch,kept=pack_group(group,tokenizer,formatter,cfg,order,spec['focus'])
    return group,order,batch,kept


def test_order(g,spec):

    return candidate_order(g,2026,reverse=spec['residual'])


def tensor_articles(group,order,device):
    return torch.as_tensor(group['articles'][order],device=device,dtype=torch.long)


@torch.no_grad()
def speed_probe(model,groups,tokenizer,formatter,cfg,spec,device):
    model.eval(); n=min(12,len(groups)); rate=[]; dim=None

    by_length=sorted(groups,key=lambda g:sum(len(x) for x in g['docs']))
    chosen=[by_length[i] for i in np.linspace(0,len(by_length)-1,n,dtype=int)]
    for j,g in enumerate(chosen):
        torch.cuda.synchronize(); t=time.monotonic()
        _,order,batch,_=prepare_one(g,tokenizer,formatter,cfg,spec,test_order(g,spec))
        docs,query=embeddings(model,batch,device); dim=docs.shape[-1]
        torch.cuda.synchronize(); elapsed=time.monotonic()-t
        if j>0 or n==1: rate.append(elapsed)
    return float(np.mean(rate)),int(dim)


def activation_probe(model,tokenizer,formatter,groups,cfg,spec,device,root,deadline):


    chosen=sorted(groups,key=lambda g: min(cfg.max_length,sum(min(cfg.doc_tokens,len(d)) for d in g['docs'])),reverse=True)[:2]
    if len(chosen)<2 or deadline-time.monotonic()<300: return {'checkpointing':True,'reason':'budget'}
    batches=[prepare_one(g,tokenizer,formatter,cfg,spec,np.arange(len(g['docs']))) for g in chosen]
    cpu_rng=torch.get_rng_state(); cuda_rng=torch.cuda.get_rng_state(device)
    result={}; total=torch.cuda.get_device_properties(device).total_memory
    model.train()
    def run(enabled):
        if enabled: model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant':False})
        else: model.gradient_checkpointing_disable()
        timings=[]; peak=0
        for j,(g,order,batch,_) in enumerate(batches):
            model.zero_grad(set_to_none=True); gc.collect(); torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats(device); torch.cuda.synchronize(device); t=time.monotonic()
            labels=torch.as_tensor(g['labels'][order],device=device); arts=tensor_articles(g,order,device)
            scores,_=forward_scores(model,None,batch,arts,device)
            loss=hard_list_loss(scores,labels,arts,cfg.temperature) if spec.get('short_loss',False) else ranking_loss(scores,labels,cfg.temperature)
            (loss*128.).backward(); torch.cuda.synchronize(device)
            if not torch.isfinite(loss).item() or not any(p.grad is not None for p in model.parameters() if p.requires_grad): raise RuntimeError('Invalid speed probe gradient')
            timings.append(time.monotonic()-t); peak=max(peak,torch.cuda.max_memory_reserved(device)); del loss,scores
        return {'seconds':float(np.mean(timings)),'peak_reserved_gib':peak/2**30,'safe':bool(peak+2.0*2**30<total)}
    try:
        result['on']=run(True)
        try:
            torch.set_rng_state(cpu_rng); torch.cuda.set_rng_state(cuda_rng,device)
            result['off']=run(False)
        except RuntimeError as exc:

            result['off']={'safe':False,'reason':str(exc)[:500]}
    finally:
        model.zero_grad(set_to_none=True); gc.collect(); torch.cuda.empty_cache()
        torch.set_rng_state(cpu_rng); torch.cuda.set_rng_state(cuda_rng,device)
        off=result.get('off',{}); on=result.get('on',{})
        enabled=not (off.get('safe',False) and off.get('seconds',float('inf'))<.95*on.get('seconds',0))
        if enabled: model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant':False})
        else: model.gradient_checkpointing_disable()
        result['checkpointing']=enabled
        atomic_json(root/'activation_probe.json',result)
    return result


def train_model(model,head,tokenizer,formatter,groups,cfg,spec,device,root,deadline):
    mixed=[g for g in groups if 0<g['labels'].sum()<len(g['labels'])]
    if not mixed: raise RuntimeError('No mixed groups')

    probe={'checkpointing':True,'reason':'safe default; activation probe disabled'}
    if os.getenv('V11_ACTIVATION_PROBE','0')=='1':
        try:
            probe=activation_probe(model,tokenizer,formatter,mixed,cfg,spec,device,root,deadline)
        except (RuntimeError,ValueError) as exc:
            model.zero_grad(set_to_none=True); gc.collect(); torch.cuda.empty_cache()
            model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant':False})
            probe={'checkpointing':True,'reason':repr(exc)}
    atomic_json(root/'activation_probe.json',probe)
    rng=np.random.default_rng(spec['seed']); plan=[]
    for i in rng.permutation(len(mixed)):
        g=mixed[i]
        if spec.get('short_loss',False) and rng.random()<.5:
            order=np.arange(len(g['docs'])); order=order[::-1].copy() if rng.random()<.5 else order
        else: order=rng.permutation(len(g['docs']))
        plan.append((g,order))
    blocks=[plan[i:i+cfg.accum] for i in range(0,len(plan),cfg.accum)]
    opt,sched,parameters=make_optimizer(model,head,cfg,len(blocks)); scaler=torch.amp.GradScaler('cuda',enabled=True,init_scale=1024.)
    def prepare(block): return [prepare_one(g,tokenizer,formatter,cfg,spec,order) for g,order in block]
    model.train()
    if head is not None: head.train()
    updates=seen=0; loss_sum=0.; block_times=[]; start=time.monotonic()
    with ThreadPoolExecutor(max_workers=1) as pool:
        pending=pool.submit(prepare,blocks[0])
        for i in tqdm(range(len(blocks)),desc='Single epoch',mininterval=10):

            margin=max(30.,1.5*max(block_times[-8:],default=45.))
            if time.monotonic()+margin>=deadline:
                print('Training time limit reached before next block',i,flush=True); break
            t=time.monotonic(); block=pending.result()

            if time.monotonic()+margin>=deadline: break
            pending=pool.submit(prepare,blocks[i+1]) if i+1<len(blocks) else None
            def backward_block(items):
                opt.zero_grad(set_to_none=True); losses=[]
                for g,order,batch,_ in items:
                    arts=tensor_articles(g,order,device); labels=torch.as_tensor(g['labels'][order],device=device)
                    scores,_=forward_scores(model,head,batch,arts,device)
                    loss=hard_list_loss(scores,labels,arts,cfg.temperature) if spec.get('short_loss',False) else ranking_loss(scores,labels,cfg.temperature)
                    if not loss.requires_grad: raise RuntimeError('Loss has no gradient')
                    if not torch.isfinite(loss).item(): raise FloatingPointError('Non-finite list loss')
                    scaler.scale(loss/len(items)).backward(); losses.append(loss.detach())
                return losses
            losses=None; last_error=None
            for attempt,limit in enumerate(dict.fromkeys([cfg.max_length,min(cfg.max_length,3072),min(cfg.max_length,2048)])):
                try:
                    items=block if attempt==0 else [prepare_one(g,tokenizer,formatter,replace(cfg,max_length=limit,doc_tokens=min(cfg.doc_tokens,384 if limit>=3072 else 256)),spec,order) for g,order,_,_ in block]
                    losses=backward_block(items)
                    if attempt: print('TRAINING_BLOCK_RECOVERED',i,'context',limit,flush=True)
                    break
                except (torch.cuda.OutOfMemoryError,FloatingPointError) as exc:
                    last_error=repr(exc)
                opt.zero_grad(set_to_none=True); gc.collect(); torch.cuda.empty_cache()
                if not probe['checkpointing']:
                    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant':False})
                    probe['checkpointing']=True; probe['fallback_during_training']=True
                    atomic_json(root/'activation_probe.json',probe)
                if time.monotonic()+margin>=deadline: break
            if losses is None:

                print('TRAINING_BLOCK_FAILED',i,last_error,flush=True)
                if updates>0: break
                raise RuntimeError(f'No successful training update: {last_error}')
            scaler.unscale_(opt); norm=torch.nn.utils.clip_grad_norm_(parameters,1.0)
            old=scaler.get_scale(); scaler.step(opt); scaler.update()
            if scaler.get_scale()>=old:
                sched.step(); updates+=1
            opt.zero_grad(set_to_none=True)
            batch_loss=float(torch.stack(losses).sum().cpu()); seen+=len(block); loss_sum+=batch_loss
            if not math.isfinite(batch_loss): raise FloatingPointError('Non-finite training loss')
            block_times.append(time.monotonic()-t)
            if (i+1)%cfg.checkpoint_every==0:
                save_checkpoint(model,head,root,dict(seen=seen,updates=updates,loss=loss_sum/seen))
    if updates<1: raise RuntimeError('No successful supervised optimizer updates')
    info=dict(seen_queries=seen,mixed_queries=len(mixed),optimizer_updates=updates,loss=loss_sum/seen,training_minutes=(time.monotonic()-start)/60,full_epoch=seen==len(mixed))
    save_checkpoint(model,head,root,info); atomic_json(root/'training.json',info)
    print('TRAINING COMPLETE',info,flush=True); return info


def hard_list_loss(scores,labels,articles,temperature=.1):
    z=scores.float()/temperature
    p=torch.where(labels>0)[0]; q=torch.where(labels==0)[0]
    if not len(p) or not len(q): return z.sum()*0
    with torch.no_grad():
        order=z.argsort(descending=True,stable=True)
        ranks=torch.empty_like(order); ranks[order]=torch.arange(len(order),device=z.device)
        discounts=1/torch.log2(ranks.float()+2)
        discounts[ranks>=10]=0
        w=(discounts[p,None]-discounts[q][None,:]).abs()
        w=(.5+.5*w/w.mean().clamp_min(1e-8))
        w=w*(1+.20*(articles[p,None]==articles[q][None,:]).float())
    pair=(w*F.softplus(z[q][None,:]-z[p,None])).sum()/w.sum().clamp_min(1e-8)
    multi=F.softplus(torch.logsumexp(z[q],0)-z[p]).mean()
    return .8*pair+.2*multi


def take_group(g, ix):
    ix=np.asarray(ix,dtype=int)
    assert len(np.unique(ix))==len(ix) and len(ix)>0
    h=dict(g)
    for k in ('docs','prefixes','hits'):
        h[k]=[g[k][i] for i in ix]
    for k in ('positions','articles','passage'):
        h[k]=np.asarray(g[k])[ix]
    h['labels']=None if g['labels'] is None else np.asarray(g['labels'])[ix]
    return h


def lexical_hard_scores(g, tokenizer):

    terms=set().union(*(set(h) for h in g['hits']))
    df={t:sum(t in h for h in g['hits']) for t in terms}
    n=len(g['docs']); qset=set(tokenizer.encode(g['query'],add_special_tokens=False))
    lex=[]
    for d,h in zip(g['docs'],g['hits']):
        idf=sum(math.log1p(n/max(1,df[t])) for t in h)
        overlap=len(set(d)&qset)/max(1,len(qset))
        lex.append(idf+overlap)
    return np.asarray(lex,dtype=float)


def shortlist_train(g,teacher,k=SHORT_K,lexical=None):

    y=np.asarray(g['labels']); t=np.asarray(teacher,dtype=float)
    assert len(t)==len(y) and np.isfinite(t).all()
    if not (0<y.sum()<len(y)): return None
    if lexical is not None:
        t=.80*zscore(t)+.20*zscore(lexical)
    order=np.argsort(-t,kind='stable'); take=order[:min(k,len(y))].copy()
    if not y[take].any():
        take[-1]=order[y[order]>0][0]
    if y[take].all():
        take[-1]=order[y[order]==0][0]

    arts=np.asarray(g['articles']); posarts=set(arts[take[y[take]>0]])
    if not any(a in posarts for a in arts[take[y[take]==0]]):
        harder=[i for i in order if i not in take and y[i]==0 and arts[i] in posarts]
        old=np.flatnonzero(y[take]==0)
        if harder and len(old)>=2: take[old[-1]]=harder[0]

    if y[take].sum()==1 and y.sum()>1:
        missing=[i for i in order if i not in take and y[i]>0]
        old=np.flatnonzero(y[take]==0)
        if missing and len(old)>=3:

            choices=[j for j in old if arts[take[j]] not in posarts]
            j=choices[-1] if choices else old[-1]
            take[j]=missing[0]
    assert len(set(take))==len(take) and 0<y[take].sum()<len(take)
    return take_group(g,take)


def stable_order(ids, scores):
    return np.lexsort((np.asarray(ids,dtype=str),-np.asarray(scores,dtype=float)))


def zscore(a):
    a=np.asarray(a,dtype=float)
    return (a-a.mean())/max(float(a.std()),1e-8)


def uncertainty_features(ids,first,second):

    a=zscore(first); available=np.isfinite(second).all()
    b=zscore(second) if available else a
    joint=.5*(a+b) if available else a
    order=stable_order(ids,joint); gaps=joint[order][:-1]-joint[order][1:]
    ia=stable_order(ids,a); ib=stable_order(ids,b)
    ra=np.empty(len(a),int); rb=np.empty(len(a),int)
    ra[ia]=np.arange(len(a)); rb[ib]=np.arange(len(a))
    top=np.union1d(ia[:5],ib[:5])
    return dict(gap12=float(gaps[0]) if len(gaps) else 10.,
        gap3=float(np.mean(gaps[:3])) if len(gaps) else 10.,
        rank_disagree=float(np.mean(np.abs(ra[top]-rb[top]))) if available else np.nan,
        score_disagree=float(np.mean(np.abs(a-b))) if available else np.nan)


def adaptive_shortlists(groups,target,first,second,base):


    rows=[]
    for g in groups:
        p=g['positions']; ids=target.id.iloc[p].to_numpy(str)
        rows.append(uncertainty_features(ids,first[p],second[p]))
    if not rows: return [],np.array([]),np.array([])
    d=pd.DataFrame(rows)
    r=pd.DataFrame({'gap12':d.gap12.rank(pct=True,ascending=False),
        'gap3':d.gap3.rank(pct=True,ascending=False),
        'rd':d.rank_disagree.rank(pct=True), 'sd':d.score_disagree.rank(pct=True)})
    w=np.array([.35,.25,.25,.15]); vals=r.to_numpy(float)
    risk=np.nansum(vals*w,axis=1)/np.sum(np.isfinite(vals)*w,axis=1)
    eligible=[i for i,g in enumerate(groups) if len(g['docs'])>SHORT_K]
    hard=set(sorted(eligible,key=lambda i:(-risk[i],groups[i]['query_id']))[:math.ceil(HARD_FRACTION*len(eligible))])
    shorts=[]; weights=[]
    for i,g in enumerate(groups):
        p=g['positions']; k=SHORT_K_HARD if i in hard else SHORT_K
        ix=stable_order(target.id.iloc[p],base[p])[:min(k,len(p))]
        shorts.append(take_group(g,ix)); weights.append(.25+.35*risk[i])
    return shorts,np.asarray(weights),risk


def cascade_scores(frame,base,short,mask,weight):

    result=np.empty(len(frame),dtype=float)
    for pos in frame.groupby('query_id',sort=False).indices.values():
        ids=frame.id.iloc[pos].to_numpy(str)
        order=stable_order(ids,base[pos]); shortlisted=order[mask[pos][order]]
        if len(shortlisted):
            assert np.isfinite(short[pos[shortlisted]]).all()
            w=float(weight if np.ndim(weight)==0 else weight[pos[0]])
            if not 0<=w<=1: raise ValueError('Invalid cascade weight')
            s=(1-w)*zscore(base[pos[shortlisted]])+w*zscore(short[pos[shortlisted]])
            new=shortlisted[stable_order(ids[shortlisted],s)]
            order[mask[pos][order]]=new
        result[pos[order]]=1-np.arange(len(pos),dtype=float)/(len(pos)+1)
    return result


def rank_vector(frame,scores):
    result=np.empty(len(frame),dtype=int)
    for pos in frame.groupby('query_id',sort=False).indices.values():
        order=stable_order(frame.id.iloc[pos],scores[pos])
        result[pos[order]]=np.arange(1,len(pos)+1)
    return result


def finite_inference(model,g,order,batch,tokenizer,formatter,cfg,spec,device):
    last=None
    for attempt,limit in enumerate(dict.fromkeys([cfg.max_length,min(cfg.max_length,3072),min(cfg.max_length,2048)])):
        try:
            if attempt:
                reduced=replace(cfg,max_length=limit,doc_tokens=min(cfg.doc_tokens,384 if limit>=3072 else 256))
                _,_,batch,_=prepare_one(g,tokenizer,formatter,reduced,spec,order)
            score,_=forward_scores(model,None,batch,tensor_articles(g,order,device),device)
            vals=score.detach().float().cpu().numpy()
            if vals.shape!=(len(order),) or not np.isfinite(vals).all():
                raise FloatingPointError('Non-finite or invalid model scores')
            if attempt: print('INFERENCE_RECOVERED',g['query_id'],'context',limit,flush=True)
            return vals
        except (torch.cuda.OutOfMemoryError,FloatingPointError) as exc:
            last=repr(exc)
        gc.collect(); torch.cuda.empty_cache()
    raise RuntimeError(f'Cannot score query {g["query_id"]}: {last}')


@torch.no_grad()
def infer_pass(model,groups,tokenizer,formatter,cfg,spec,device,n_rows,deadline,
               ordering,root,name,allow_partial=False):


    model.eval()
    scores=np.full(n_rows,np.nan); retained=np.zeros(n_rows,dtype=int)
    lens=[]
    def prepare(g):
        if ordering=='canonical': order=np.arange(len(g['docs']))
        elif ordering=='reverse': order=np.arange(len(g['docs']))[::-1].copy()
        elif ordering=='article': order=candidate_order(g,2026)
        elif ordering=='article_reverse': order=candidate_order(g,2026,reverse=True)
        else: raise ValueError(ordering)
        return prepare_one(g,tokenizer,formatter,cfg,spec,order)
    start=time.monotonic()
    with ThreadPoolExecutor(max_workers=1) as pool:
        pending=pool.submit(prepare,groups[0]) if groups else None
        for i in tqdm(range(len(groups)),desc=name,mininterval=20):
            if time.monotonic()+15>=deadline:
                if allow_partial: break
                raise TimeoutError(f'{name}: required inference missed deadline')
            has_next_prepared=False
            try:
                g,order,batch,kept=pending.result()
                pending=pool.submit(prepare,groups[i+1]) if i+1<len(groups) else None
                has_next_prepared=True
                values=finite_inference(model,g,order,batch,tokenizer,formatter,cfg,spec,device)
            except (RuntimeError,ValueError,FloatingPointError) as exc:
                if not allow_partial: raise

                print('OPTIONAL_PASS_SKIPPED',name,groups[i]['query_id'],repr(exc),flush=True)

                if not has_next_prepared:
                    pending=pool.submit(prepare,groups[i+1]) if i+1<len(groups) else None
                continue
            ix=g['positions'][order]
            if np.isfinite(scores[ix]).any(): raise RuntimeError('Duplicate prediction')
            scores[ix]=values
            retained[g['positions']]=kept
            lens.append(batch['input_ids'].shape[1])
            if (i+1)%200==0:
                save_array(root/(name+'.partial.npy'),scores)
    expected=np.concatenate([g['positions'] for g in groups]) if groups else np.array([],int)
    completed=int(sum(np.isfinite(scores[g['positions']]).all() for g in groups))
    if not allow_partial and (not np.isfinite(scores[expected]).all()):
        raise RuntimeError(f'{name}: missing predictions')
    save_array(root/(name+'.npy'),scores)
    atomic_json(root/(name+'.json'),dict(queries=completed,expected=len(groups),
        minutes=(time.monotonic()-start)/60,token_p50=float(np.median(lens)) if lens else None))
    print(name,completed,'/',len(groups),'minutes',(time.monotonic()-start)/60,flush=True)
    return scores,retained


def mix_passes(first,second):


    return np.where(np.isfinite(second),.5*(first+second),first)


def merge_part(dst,part):
    finite=np.isfinite(part)
    if np.isfinite(dst[finite]).any(): raise RuntimeError('Duplicate chunk prediction')
    dst[finite]=part[finite]


def target_chunks(groups):

    ordered=sorted(groups,key=lambda g:hashlib.blake2b(g['query_id'].encode(),digest_size=8).digest())
    return [ordered[i:i+CHUNK_SIZE] for i in range(0,len(ordered),CHUNK_SIZE)]


def coverage(second,mask,n_val):
    def ratio(a,m): return float(np.isfinite(a[m]).mean()) if m.any() else None
    return dict(second_coverage=ratio(second,mask),
        second_validation_coverage=ratio(second[:n_val],mask[:n_val]),
        second_test_coverage=ratio(second[n_val:],mask[n_val:]))


def worker(branch,root):
    cfg=CFG; spec=BRANCHES[branch].copy(); local=root/branch; local.mkdir(exist_ok=True)
    payload=pickle.load(open(root/'payload.pkl','rb')); fit=payload['fit']; target=payload['target']
    deadline=payload['deadline']; start=payload['start']; n_val=payload['n_val']
    torch.set_num_threads(2); device=torch.device('cuda:0'); torch.cuda.set_device(device)
    model,tokenizer,formatter=make_model(cfg,spec,device)
    target_groups=prepare_groups(target,tokenizer,cfg,spec['focus'])
    fit_groups=prepare_groups(fit,tokenizer,cfg,spec['focus'])
    chunks=target_chunks(target_groups); n=len(target)
    if branch=='full':
        rate,_=speed_probe(model,target_groups,tokenizer,formatter,cfg,spec,device)

        train_deadline=min(start+165*60,deadline-(2*max(rate,1.15)*len(target_groups)*1.18+22*60))
        if train_deadline-time.monotonic()<120: raise TimeoutError('Insufficient full training budget')
        info=train_model(model,None,tokenizer,formatter,fit_groups,cfg,spec,device,local,train_deadline)
        del fit_groups; gc.collect(); torch.cuda.empty_cache()
        first_all=np.full(n,np.nan); second_all=np.full(n,np.nan); done=0; elapsed=0.
        for i,chunk in enumerate(chunks):
            prefix=f'chunk{i:03d}'
            tick=time.monotonic()
            first,_=infer_pass(model,chunk,tokenizer,formatter,cfg,spec,device,n,
                         deadline-18*60,'canonical',local,prefix+'_first')
            elapsed+=time.monotonic()-tick; done+=len(chunk)
            merge_part(first_all,first)

            remaining=len(target_groups)-done
            future=remaining*max(rate,elapsed/max(1,done),1.10)*1.20
            rev_deadline=deadline-18*60-future
            second,_=infer_pass(model,chunk,tokenizer,formatter,cfg,spec,device,n,
                         rev_deadline,'reverse',local,prefix+'_reverse',allow_partial=True)
            merge_part(second_all,second)
            atomic_json(local/(prefix+'_ready.json'),dict(complete=True,queries=[g['query_id'] for g in chunk]))
        base=mix_passes(first_all,second_all); assert np.isfinite(base).all()
        save_array(local/'first.npy',first_all); save_array(local/'reverse.npy',second_all); save_array(local/'scores.npy',base)
        write_submission(target.iloc[n_val:].reset_index(drop=True),first_all[n_val:],root/'submission_full_first.csv')
        write_submission(target.iloc[n_val:].reset_index(drop=True),base[n_val:],root/'submission_full_tta.csv')
        atomic_json(local/'status.json',dict(complete=True,**info,**coverage(second_all,np.ones(n,bool),n_val),
                    peak_gpu_gib=torch.cuda.max_memory_allocated()/2**30))


        try:
            def difficulty(g):
                p=g['positions']; f=uncertainty_features(target.id.iloc[p],first_all[p],second_all[p]);
                return 1/(.15+f['gap12']) + .5/(.20+f['gap3']) + .3*np.nan_to_num(f['rank_disagree'],nan=0.)
            priority=sorted(target_groups,key=lambda g:(-difficulty(g),g['query_id']))[:1200]
            if time.monotonic()+90<deadline-10*60:
                alt_cfg=replace(cfg,max_length=5120,doc_tokens=640)
                extra,_=infer_pass(model,priority,tokenizer,formatter,alt_cfg,spec,device,n,
                        deadline-10*60,'article',local,'extra',allow_partial=True)
                extra_stats=coverage(extra,np.ones(n,bool),n_val)
                atomic_json(local/'extra_coverage.json',extra_stats)
                print('Optional extra coverage:',extra_stats,flush=True)
        except Exception as exc:
            print('OPTIONAL_EXTRA_FAILED',repr(exc),flush=True)
    else:

        miner_cfg=replace(cfg,max_length=2560,doc_tokens=320); mine_spec=dict(spec,focus=False)
        mine,_=infer_pass(model,fit_groups,tokenizer,formatter,miner_cfg,mine_spec,device,len(fit),
                     min(start+58*60,deadline-150*60),'canonical',local,'frozen_teacher',allow_partial=True)
        count=0; shorts=[]; widened=0; repairs=0
        for g in fit_groups:
            p=g['positions']; values=mine[p]; lexical=lexical_hard_scores(g,tokenizer)
            if not np.isfinite(values).all(): values=lexical
            else: count+=1

            u=int.from_bytes(hashlib.blake2b(g['query_id'].encode(),digest_size=8).digest(),'little')/2**64
            k=SHORT_K_HARD if u<HARD_FRACTION else SHORT_K
            order=np.argsort(-(.8*zscore(values)+.2*zscore(lexical)),kind='stable')[:min(k,len(values))]
            r=g['labels'][order]; repairs+=int(not r.any() or r.all())
            h=shortlist_train(g,values,k,lexical)
            if h is not None:
                shorts.append(h); widened+=int(len(h['docs'])>SHORT_K)
        if not shorts: raise RuntimeError('No supervised short lists')
        mining=dict(neural_queries=count,total_queries=len(fit_groups),initial_homogeneous=repairs,
                    short_groups=len(shorts),wide_groups=widened)
        atomic_json(local/'mining.json',mining)
        del fit_groups; gc.collect(); torch.cuda.empty_cache()
        short_cfg=replace(cfg,doc_tokens=640,max_length=4096,accum=6); spec['short_loss']=True
        short_deadline=min(start+158*60,deadline-85*60)
        if short_deadline-time.monotonic()<120: raise TimeoutError('Insufficient short training budget')
        info=train_model(model,None,tokenizer,formatter,shorts,short_cfg,spec,device,local,short_deadline)
        del shorts; gc.collect(); torch.cuda.empty_cache()
        first_all=np.full(n,np.nan); second_all=np.full(n,np.nan); mask=np.zeros(n,bool)
        weights=np.full(n,np.nan); risks=np.full(n,np.nan); all_short=[]

        for i,chunk in enumerate(chunks):
            prefix=f'chunk{i:03d}'; pub=root/'full'/(prefix+'_ready.json')
            while not pub.is_file():
                if time.monotonic()>=deadline-12*60: raise TimeoutError('Missing full-list chunk')
                time.sleep(2)
            ready=read_json(pub)
            if ready.get('queries')!=[g['query_id'] for g in chunk]: raise RuntimeError('Chunk order mismatch')
            a=np.load(root/'full'/(prefix+'_first.npy')); b=np.load(root/'full'/(prefix+'_reverse.npy'))
            base=mix_passes(a,b)
            selected,w,r=adaptive_shortlists(chunk,target,a,b,base)
            for g,h,ww,rr in zip(chunk,selected,w,r):
                mask[h['positions']]=True; weights[g['positions']]=ww; risks[g['positions']]=rr
            part,_=infer_pass(model,selected,tokenizer,formatter,short_cfg,spec,device,n,
                         deadline-8*60,'canonical',local,prefix+'_first')
            merge_part(first_all,part)
            all_short.extend(selected)

            save_array(local/'first_partial.npy',first_all)
        assert np.isfinite(first_all[mask]).all() and np.isfinite(weights).all()
        save_array(local/'first.npy',first_all); save_array(local/'mask.npy',mask)
        save_array(local/'weights.npy',weights); save_array(local/'risk.npy',risks)


        collected=np.full(n,np.nan)
        for i in range(len(chunks)):
            prefix=f'chunk{i:03d}'
            aa=np.load(root/'full'/(prefix+'_first.npy')); bb=np.load(root/'full'/(prefix+'_reverse.npy'))
            merge_part(collected,mix_passes(aa,bb))
        interim=cascade_scores(target,collected,first_all,mask,weights)
        write_submission(target.iloc[n_val:].reset_index(drop=True),interim[n_val:],root/'submission_cascade_required.csv')
        write_submission(target.iloc[n_val:].reset_index(drop=True),interim[n_val:],Path('submission.csv'))

        priority=sorted(all_short,key=lambda g:(-risks[g['positions'][0]],g['query_id']))
        second_all,_=infer_pass(model,priority,tokenizer,formatter,short_cfg,spec,device,n,
                          deadline-12*60,'reverse',local,'reverse',allow_partial=True)
        full=mix_passes(first_all,second_all); assert np.isfinite(full[mask]).all()
        save_array(local/'scores.npy',full)
        atomic_json(local/'status.json',dict(complete=True,**info,neural_mining_queries=count,
                    **coverage(second_all,mask,n_val),peak_gpu_gib=torch.cuda.max_memory_allocated()/2**30))
    print('COMPLETE',branch,flush=True)


def validation_report(held,channels,root,components,n_val):
    scores={k:v[:n_val] for k,v in channels.items()}
    ranks={k:rank_vector(held,v) for k,v in scores.items()}
    qids=list(held.groupby('query_id',sort=True).indices)
    table=pd.DataFrame(dict(query_id=qids,component=[components[q] for q in qids]))
    for k,s in scores.items():
        table[k]=ndcg_vector(held,s)
        table[k+'_top1']=[int(held.relevance.iloc[pos][ranks[k][pos]==1].iloc[0])
                           for pos in held.groupby('query_id',sort=True).indices.values()]
    table['delta']=table['cascade']-table['full_tta']
    table['final_delta']=table['final']-table['full_tta']
    table['tta_delta']=table['full_tta']-table['full_first']
    table.to_csv(root/'validation_per_query.csv',index=False)
    stats={k:float(table[k].mean()) for k in table if k not in ('query_id','component')}
    agg=table.groupby('component').delta.agg(['sum','size']).to_numpy()
    rng=np.random.default_rng(2026)
    boots=[]
    for _ in range(3000):
        a=agg[rng.integers(0,len(agg),len(agg))]
        boots.append(a[:,0].sum()/a[:,1].sum())
    stats['cascade_delta_ci95']=np.percentile(boots,[2.5,97.5]).tolist()
    stats['final_delta_ci95_descriptive']=paired_ci(table,table.final_delta.to_numpy())
    stats['improved_queries']=int((table.delta>1e-12).sum())
    stats['worse_queries']=int((table.delta< -1e-12).sum())
    stats['final_full_validation_descriptive']=True
    if (root/'calibration.json').exists():
        cal=read_json(root/'calibration.json'); stats['selected']=cal.get('selected'); stats['calibration_oof_ndcg']=cal.get('calibration_oof_ndcg'); stats['verification_ndcg']=cal.get('verification_ndcg'); stats['pooled_oof_ndcg']=cal.get('pooled_oof_ndcg'); stats['exceeds_historical_best']=cal.get('exceeds_historical_best')
    atomic_json(root/'metrics.json',stats)
    return stats


def paired_ci(table,delta,seed=2026,n_boot=2000,alpha=.05):
    agg=pd.DataFrame({'component':table['component'],'delta':delta}).groupby('component').delta.agg(['sum','size']).to_numpy()
    if not len(agg): return [None,None]
    rng=np.random.default_rng(seed); out=[]
    for _ in range(n_boot):
        a=agg[rng.integers(0,len(agg),len(agg))]; out.append(a[:,0].sum()/a[:,1].sum())
    return np.quantile(out,[alpha/2,1-alpha/2]).tolist()


def guarded_weights(frame,base,short,mask,weights):
    out=np.asarray(weights,dtype=float).copy()
    for pos in frame.groupby('query_id',sort=False).indices.values():
        ix=pos[mask[pos]]
        if len(ix)<2: continue
        vals=zscore(base[ix]); ss=zscore(short[ix]); order=stable_order(frame.id.iloc[ix],vals)
        if vals[order[0]]-vals[order[1]]>=1.0 and ss[order[0]]<ss.max():
            out[pos]=min(float(out[pos[0]]),.15)
    return out


def final_calibration(held,target,channels,base,short,mask,weights,components,n_val,root):


    qids=list(held.groupby('query_id',sort=True).indices); comp=np.array([components[q] for q in qids]); uniq=np.unique(comp)
    rng=np.random.default_rng(17711); shuffled=rng.permutation(uniq); cal_set=set(shuffled[:len(shuffled)//2]); cal=np.isin(comp,list(cal_set)); audit=~cal
    if min(cal.sum(),audit.sum())<60: raise RuntimeError('Too few independent queries for calibration')
    tab=pd.DataFrame({'query_id':qids,'component':comp,'split':np.where(cal,'calibration','audit')})
    alt_names=['cascade','cascade_fixed45','cascade_strong','cascade_light','cascade_flat35','cascade_flat55','cascade_guarded']
    channels['cascade_light']=cascade_scores(target,base,short,mask,.20)
    channels['cascade_flat35']=cascade_scores(target,base,short,mask,.35)
    channels['cascade_flat55']=cascade_scores(target,base,short,mask,.55)


    guard=guarded_weights(target,base,short,mask,weights)
    channels['cascade_guarded']=cascade_scores(target,base,short,mask,guard)
    for k in alt_names+['full_tta','full_first']:
        tab[k]=ndcg_vector(held,channels[k][:n_val])
    selected='cascade'; evidence=[]
    for k in alt_names[1:]:
        delta=tab.loc[cal,k].to_numpy()-tab.loc[cal,'cascade'].to_numpy()
        ci=paired_ci(tab.loc[cal],delta,alpha=.05/max(1,len(alt_names)-1))
        evidence.append({'variant':k,'calibration_ndcg':float(tab.loc[cal,k].mean()),'delta':float(delta.mean()),'ci_bonferroni':ci,'eligible':bool(delta.mean()>=.001 and ci[0]>0)})
    eligible=[x for x in evidence if x['eligible']]
    if eligible: selected=max(eligible,key=lambda x:x['calibration_ndcg'])['variant']
    channels['final']=channels[selected].copy(); tab['final']=tab[selected]
    stats=dict(selected=selected,calibration_queries=int(cal.sum()),audit_queries=int(audit.sum()),
      calibration_ndcg=float(tab.loc[cal,'final'].mean()),audit_ndcg=float(tab.loc[audit,'final'].mean()),
      audit_baseline_full=float(tab.loc[audit,'full_tta'].mean()),audit_fixed_cascade=float(tab.loc[audit,'cascade'].mean()),
      audit_delta_vs_fixed_ci95=paired_ci(tab.loc[audit],tab.loc[audit,'final'].to_numpy()-tab.loc[audit,'cascade'].to_numpy()),
      audit_delta_vs_full_ci95=paired_ci(tab.loc[audit],tab.loc[audit,'final'].to_numpy()-tab.loc[audit,'full_tta'].to_numpy()),candidates=evidence)
    tab.to_csv(root/'calibration_per_query.csv',index=False); atomic_json(root/'calibration.json',stats)
    return stats


def detailed_diagnostics(held,channels,base,short,mask,weights,risk,n_val,root):


    rows=[]; scores={k:v[:n_val] for k,v in channels.items()}; br=rank_vector(held,base[:n_val]); fr=rank_vector(held,scores['final'])
    def dcg(y):
        return float(np.sum(y[:10]/np.log2(np.arange(2,min(10,len(y))+2))))
    for q,pos in held.groupby('query_id',sort=True).indices.items():
        y=held.relevance.iloc[pos].to_numpy(int); ids=held.id.iloc[pos].to_numpy(str); order=stable_order(ids,base[pos]); slots=np.flatnonzero(mask[pos][order]); oracle=y[order].copy(); oracle[slots]=np.sort(oracle[slots])[::-1]
        ideal=dcg(np.sort(y)[::-1]); f_order=stable_order(ids,scores['final'][pos]); n0=dcg(y[order])/ideal if ideal else 0.; n1=dcg(y[f_order])/ideal if ideal else 0.
        arts=held.article_id.iloc[pos].to_numpy(str); mixed=any(len(set(y[arts==a]))>1 for a in set(arts));
        rows.append(dict(query_id=q,n_candidates=len(pos),positives=int(y.sum()),shortlist_size=int(mask[pos].sum()),shortlist_positives=int(y[mask[pos]].sum()),
            mixed_article=mixed,risk=float(risk[pos[0]]),weight=float(weights[pos[0]]),base_ndcg=n0,final_ndcg=n1,delta=n1-n0,
            oracle_ndcg=dcg(oracle)/ideal if ideal else 0.,base_top1=int(y[order[0]]),final_top1=int(y[f_order[0]]),
            positive_first_rank=int(br[pos][y>0].min()) if y.any() else -1,final_positive_first_rank=int(fr[pos][y>0].min()) if y.any() else -1,
            same_article_top1_error=bool(y[order[0]]==0 and np.any((arts==arts[order[0]]) & (y==1))),
            short_margin=float(np.sort(short[pos][mask[pos]])[-1]-np.sort(short[pos][mask[pos]])[-2]) if mask[pos].sum()>1 else 0.))
    d=pd.DataFrame(rows); d['risk_quartile']=pd.qcut(d.risk.rank(method='first'),4,labels=['Q1','Q2','Q3','Q4']); d['oracle_headroom']=d.oracle_ndcg-d.base_ndcg
    d.to_csv(root/'validation_cascade_details.csv',index=False)
    segments=[]
    for feature in ['risk_quartile','shortlist_size','mixed_article','positives']:
        for name,g in d.groupby(feature,observed=True):
            segments.append(dict(segment=feature,value=str(name),queries=len(g),base_ndcg=g.base_ndcg.mean(),final_ndcg=g.final_ndcg.mean(),delta=g.delta.mean(),oracle_headroom=g.oracle_headroom.mean(),base_top1=g.base_top1.mean(),final_top1=g.final_top1.mean(),missing_positive_queries=int((g.shortlist_positives<g.positives).sum())))
    pd.DataFrame(segments).to_csv(root/'validation_segments.csv',index=False)
    atomic_json(root/'diagnostic_summary.json',dict(oracle_ndcg=float(d.oracle_ndcg.mean()),remaining_oracle_headroom=float((d.oracle_ndcg-d.final_ndcg).mean()),
      corrected_top1=int(((d.base_top1==0)&(d.final_top1==1)).sum()),broken_top1=int(((d.base_top1==1)&(d.final_top1==0)).sum()),
      no_positive_in_shortlist=int(((d.positives>0)&(d.shortlist_positives==0)).sum()),all_positives_in_shortlist=int((d.shortlist_positives==d.positives).sum())))


def main():
    start=float(os.environ.get('V13_FINAL_START',time.monotonic())); deadline=start+TOTAL_MINUTES*60-6*60
    mode=os.getenv('AMML_RUN_MODE','validate_and_submit')
    if mode not in ('validate_and_submit','full'): raise ValueError(mode)
    if torch.cuda.device_count()<2: raise RuntimeError('Select T4 x2')
    root=(Path('amml_v13_extended')/mode).resolve()

    stamp=time.strftime('%Y%m%d-%H%M%S')+'-'+str(os.getpid())
    if root.exists(): root.rename(root.with_name(root.name+'.archive-'+stamp))
    root.mkdir(parents=True,exist_ok=False)
    atomic_json(root/'run_identity.json',{'start':start,'version':PIPELINE_VERSION})
    previous=Path('submission.csv')
    if previous.exists(): previous.rename(Path('submission.previous-'+stamp+'.csv'))
    previous_zip=Path('v13_results.zip')
    if previous_zip.exists(): previous_zip.rename(Path('v13_results.previous-'+stamp+'.zip'))
    raw_train,raw_test=read_data(find_data_files()); train,test=clean_data(raw_train,raw_test)
    audit_data(raw_train,raw_test,train,test,root)
    if mode=='validate_and_submit':
        fi,vi,components=connected_query_split(train)
        fit=train.iloc[fi].reset_index(drop=True); held=train.iloc[vi].reset_index(drop=True)
        target=pd.concat([held.drop(columns='relevance'),test],ignore_index=True)
        n_val=len(held)
        print('Strict validation + test:',fit.query_id.nunique(),held.query_id.nunique(),test.query_id.nunique(),flush=True)
    else:
        fit=train; held=None; components=None; target=test; n_val=0
    assert set(fit.document_id).isdisjoint(target.document_id)
    assert fit.id.is_unique and target.id.is_unique and set(fit.query_id).isdisjoint(target.query_id)
    atomic_json(root/'runtime.json',dict(model=CFG.model,revision=CFG.revision,config=asdict(CFG),
              branches=BRANCHES,torch=torch.__version__,transformers=transformers.__version__,
              cuda=torch.version.cuda,python=sys.version,gpus=[torch.cuda.get_device_name(i) for i in range(2)]))
    payload=dict(fit=fit,target=target,deadline=deadline,start=start,mode=mode,n_val=n_val,held=held,components=components,raw_test=raw_test)
    with (root/'payload.pkl').open('wb') as f: pickle.dump(payload,f,protocol=4)
    children={}; logs={}
    visible=os.getenv('CUDA_VISIBLE_DEVICES'); devices=visible.split(',')[:2] if visible else ['0','1']
    try:
        for gpu,b in enumerate(BRANCHES):
            logs[b]=(root/f'{b}.log').open('w',buffering=1)
            env=dict(os.environ,CUDA_VISIBLE_DEVICES=devices[gpu],OMP_NUM_THREADS='2',
                     MKL_NUM_THREADS='2',OPENBLAS_NUM_THREADS='1',PYTHONUNBUFFERED='1')
            children[b]=subprocess.Popen([sys.executable,'-u',str(Path(__file__).resolve()),
                    '--worker',b,'--root',str(root)],env=env,stdout=logs[b],stderr=subprocess.STDOUT)
        offsets={b:0 for b in BRANCHES}; reported=set()
        while any(p.poll() is None for p in children.values()):
            if time.monotonic()>=deadline: break

            for b,p in children.items():
                path=root/f'{b}.log'
                with path.open('rb') as f:
                    f.seek(offsets[b]); delta=f.read(); offsets[b]=f.tell()
                if delta:
                    print(f'[{b}/{p.poll()}] '+delta.decode(errors='replace').replace('\r','\n')[-1200:],flush=True)
            failed={b:p.returncode for b,p in children.items() if p.poll() not in (None,0)}
            for b,code in failed.items():
                if b in reported: continue
                reported.add(b)
                tail=(root/f'{b}.log').read_text(encoding='utf-8',errors='replace').replace('\r','\n')[-16000:]
                print(f'WORKER_FAILED {b}: exit={code}\n{tail}',flush=True)
            if 'full' in failed and not (root/'full'/'scores.npy').is_file():

                break
            time.sleep(min(15,max(0,deadline-time.monotonic())))
    finally:
        for p in children.values():
            if p.poll() is None: p.terminate()
        for p in children.values():
            try: p.wait(timeout=8)
            except subprocess.TimeoutExpired: p.kill(); p.wait()
        for f in logs.values(): f.close()
    statuses={b:read_json(root/b/'status.json') for b in BRANCHES}
    if all(s.get('complete') and s.get('optimizer_updates',0)>0 for s in statuses.values()):
        try:
            finalize(root,raw_test,target,held,components,n_val,start,mode,statuses,fit)
            return
        except Exception:
            import traceback
            traceback.print_exc()
            print('Final diagnostics failed; attempting recovery from current-run predictions',flush=True)
    recover_current(root,raw_test,target,held,n_val,start,mode,statuses)


def strong_channel_bank(target,base,short,mask,weights,first,reverse,short_first,short_reverse,extra):
    """Predictions only, no relevance labels. V11 channels remain available unchanged."""
    channels={}; n=len(target)
    rankblend=np.empty(n); confidence=np.empty(n); robust=np.empty(n)
    extra_base=base.copy()
    for pos in target.groupby('query_id',sort=False).indices.values():
        ids=target.id.iloc[pos].to_numpy(str)
        if np.isfinite(extra[pos]).all():
            extra_base[pos]=.70*zscore(base[pos])+.30*zscore(extra[pos])
    def ranks(v,ids):
        order=stable_order(ids,v); r=np.empty(len(v),float); r[order]=np.arange(len(v)); return -r/max(1,len(v)-1)
    for pos in target.groupby('query_id',sort=False).indices.values():
        ids=target.id.iloc[pos].to_numpy(str); order=stable_order(ids,base[pos]); loc=order[mask[pos][order]]; ix=pos[loc]
        for out in (rankblend,confidence,robust): out[pos[order]]=1-np.arange(len(pos))/(len(pos)+1.)
        if len(ix)<2: continue
        ii=ids[loc]; b=zscore(base[ix]); s=zscore(short[ix]); w=float(weights[pos[0]])

        rb=zscore(ranks(b,ii)); rs=zscore(ranks(s,ii))
        r=.20*b+.45*s+.10*rb+.25*rs
        bf=zscore(first[ix]); br=zscore(reverse[ix]) if np.isfinite(reverse[ix]).all() else bf
        sf=zscore(short_first[ix]); sr=zscore(short_reverse[ix]) if np.isfinite(short_reverse[ix]).all() else sf

        bd=float(np.mean(np.abs(bf-br))) if np.isfinite(reverse[ix]).all() else .7
        sd=float(np.mean(np.abs(sf-sr))) if np.isfinite(short_reverse[ix]).all() else .7
        cw=float(np.clip(.60+.25*(bd-sd),.35,.85))
        c=(1-cw)*b+cw*s

        z=.30*(bf+br)/2+.70*(sf+sr)/2
        for out,v in ((rankblend,r),(confidence,c),(robust,z)):
            new=loc[stable_order(ii,v)]; o=order.copy(); o[mask[pos][order]]=new
            out[pos[o]]=1-np.arange(len(pos))/(len(pos)+1.)
    channels.update(rank_consensus=rankblend,confidence_mix=confidence,normalized_passes=robust,
                    extra_cascade=cascade_scores(target,extra_base,short,mask,weights),
                    extra_short=cascade_scores(target,extra_base,short,mask,.80))
    return channels


def calibration_features(target,base,short,mask,weights,first,reverse,short_first,short_reverse,extra):


    result=[]; groups=list(target.groupby('query_id',sort=True).indices.items())
    for q,pos in groups:
        ids=target.id.iloc[pos].to_numpy(str); order=stable_order(ids,base[pos]); loc=order[mask[pos][order]]; ix=pos[loc]
        if not len(ix): raise ValueError('Empty shortlist')
        b=zscore(base[ix]); s=zscore(short[ix]); w=float(weights[pos[0]])
        prior=(1-w)*b+w*s
        f=zscore(first[ix]); r=zscore(reverse[ix]) if np.isfinite(reverse[ix]).all() else f
        sf=zscore(short_first[ix]); sr=zscore(short_reverse[ix]) if np.isfinite(short_reverse[ix]).all() else sf
        e=zscore(extra[ix]) if np.isfinite(extra[ix]).all() else b
        terms=query_terms(str(target['query'].iloc[ix[0]])); den=max(1,len(terms))
        lex=[]; title=[]
        for row in target.iloc[ix].itertuples():
            dt={stem_word(m.group()) for m in WORD.finditer(str(row.text))}; tt={stem_word(m.group()) for m in WORD.finditer(str(row.title))}
            lex.append(len(terms&dt)/den); title.append(len(terms&tt)/den)
        def rg(v):
            o=stable_order(ids[loc],v); rr=np.empty(len(v)); rr[o]=np.arange(len(v)); return zscore(-rr)
        bd=float(np.mean(np.abs(f-r))) if np.isfinite(reverse[ix]).all() else .7
        sd=float(np.mean(np.abs(sf-sr))) if np.isfinite(short_reverse[ix]).all() else .7
        X=np.column_stack([s-b,rg(s)-rg(b), (s-b)*np.clip(bd-sd,-1,1), (s-b)*(w-.45)*2,
                           e-b, (sf+sr)/2-s, (f+r)/2-b, np.clip(zscore(lex),-2,2),np.clip(zscore(title),-2,2)])
        result.append((q,pos,order,loc,ix,prior,np.clip(X,-3,3)))
    return result


def fit_residual(features, labels, train_qids):


    A=[]; offset=[]; importance=[]
    for q,pos,order,loc,ix,prior,X in features:
        if q not in train_qids: continue
        y=labels[ix]; p=np.flatnonzero(y==1); neg=np.flatnonzero(y==0)
        if len(p)==0 or len(neg)==0: continue
        rr=np.empty(len(ix),int); rr[np.argsort(-prior,kind='stable')]=np.arange(len(ix)); disc=1/np.log2(rr+2.)
        weight=np.abs(disc[p,None]-disc[neg][None,:]); weight=weight/max(weight.sum(),1e-9)
        A.append((X[p,None,:]-X[neg][None,:,:]).reshape(-1,X.shape[1])); offset.append((prior[p,None]-prior[neg][None,:]).ravel()*3)
        importance.append(weight.ravel())
    if not A: return np.zeros(9)
    A=np.concatenate(A); offset=np.concatenate(offset); W=np.concatenate(importance); W/=W.sum()
    beta=np.zeros(A.shape[1]); ridge=.12

    for _ in range(20):
        z=np.clip(offset+A@beta,-35,35); prob=1/(1+np.exp(z)); grad=-(A.T@(W*prob))+ridge*beta
        H=A.T@((W*prob*(1-prob))[:,None]*A)+ridge*np.eye(len(beta))
        step=np.linalg.solve(H,grad); beta-=step
        if np.linalg.norm(step)<1e-7: break
    return np.clip(beta,-.8,.8)


def residual_prediction(target,features,beta_by_q):
    out=np.empty(len(target))
    for q,pos,order,loc,ix,prior,X in features:
        beta=beta_by_q(q); s=prior+np.clip(X@beta/3,-.65,.65)
        new=loc[stable_order(target.id.iloc[ix],s)]; o=order.copy(); o[np.isin(order,loc)]=new
        out[pos[o]]=1-np.arange(len(pos))/(len(pos)+1.)
    return out


def select_v13(held,target,channels,features,components,n_val,root):
    qids=list(held.groupby('query_id',sort=True).indices); comp=np.array([components[q] for q in qids]); uniq=np.unique(comp)

    shuffled=np.random.default_rng(17711).permutation(uniq); cal=np.isin(comp,shuffled[:len(shuffled)//2]); audit=~cal
    if min(cal.sum(),audit.sum())<60: raise RuntimeError('Too few independent queries')
    labels=np.full(len(target),-1); labels[:n_val]=held.relevance.to_numpy(); cal_q=set(np.asarray(qids)[cal])
    cal_comps=np.unique(comp[cal]); folds=np.array_split(np.random.default_rng(913).permutation(cal_comps),4)
    by_q={}; final_beta=fit_residual(features,labels,cal_q)
    for block in folds:
        valid=cal & np.isin(comp,block); train_q=set(np.asarray(qids)[cal & ~valid]); beta=fit_residual(features,labels,train_q)
        for q in np.asarray(qids)[valid]: by_q[q]=beta

    honest=residual_prediction(target,features,lambda q:by_q.get(q,final_beta))
    channels['residual_calibrated']=residual_prediction(target,features,lambda q:final_beta)
    candidates=['cascade','shortlist_only','cascade_top6_fixed45','full_tta','rank_consensus','confidence_mix','normalized_passes','extra_cascade','extra_short','residual_calibrated']
    tab=pd.DataFrame({'query_id':qids,'component':comp,'split':np.where(cal,'calibration_oof','verification')})
    for k in candidates:
        s=honest if k=='residual_calibrated' else channels[k]; tab[k]=ndcg_vector(held,s[:n_val])
    evidence=[]
    for k in candidates[1:]:
        d=tab[k]-tab['cascade']; v=tab.loc[audit,k].mean()-tab.loc[audit,'cascade'].mean()
        evidence.append(dict(variant=k,calibration_oof_ndcg=float(tab.loc[cal,k].mean()),calibration_delta=float(d[cal].mean()),verification_ndcg=float(tab.loc[audit,k].mean()),verification_delta=float(v),verification_ci95=paired_ci(tab.loc[audit],d[audit].to_numpy()), pooled_oof_ndcg=float(tab[k].mean())))
    ranked=sorted(evidence,key=lambda a:a['calibration_oof_ndcg'],reverse=True)

    best=ranked[0]; selected='cascade'
    if (best['calibration_delta']>=.0003 and best['verification_delta']>0
            and best['verification_ndcg']>=tab.loc[audit,'shortlist_only'].mean()
            and best['pooled_oof_ndcg']>0.968164123288774): selected=best['variant']
    channels['final']=channels[selected].copy(); tab['final']=tab[selected]
    stats=dict(selected=selected,reason='Fixed candidate chosen on component OOF; separate verification veto; no test labels. Verification is used in the decision, not an untouched final test.',proposed=best['variant'],calibration_queries=int(cal.sum()),verification_queries=int(audit.sum()),calibration_oof_ndcg=float(tab.loc[cal,'final'].mean()),verification_ndcg=float(tab.loc[audit,'final'].mean()),verification_cascade=float(tab.loc[audit,'cascade'].mean()),pooled_oof_ndcg=float(tab['final'].mean()), historical_v11=0.9675723833784988,historical_v11_best=0.968164123288774, exceeds_historical_best=bool(tab['final'].mean()>0.968164123288774),residual_coefficients=final_beta.tolist(),candidates=evidence)
    tab.to_csv(root/'calibration_per_query.csv',index=False); atomic_json(root/'calibration.json',stats)
    return stats


def compact_export(root):

    import zipfile
    out=Path('v13_results.zip'); tmp=out.with_suffix('.tmp.zip')
    with zipfile.ZipFile(tmp,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=3) as z:
        for p in sorted(root.glob('*')):
            if p.is_file() and p.suffix in ('.csv','.json'):
                z.write(p,p.name)
        for b in BRANCHES:
            for name in ('status.json','training.json','checkpoint.pt','extra_coverage.json','first.npy','reverse.npy','scores.npy','mask.npy','weights.npy','risk.npy','extra.npy'):
                p=root/b/name
                if p.is_file(): z.write(p,str(Path(b)/name))
        if Path('submission.csv').is_file(): z.write('submission.csv','final_submission.csv')
    tmp.replace(out)


def finalize(root,raw_test,target,held,components,n_val,start,mode,statuses,fit):
    first=np.load(root/'full'/'first.npy'); base=np.load(root/'full'/'scores.npy')
    short=np.load(root/'short'/'scores.npy'); mask=np.load(root/'short'/'mask.npy')
    weights=np.load(root/'short'/'weights.npy'); risk=np.load(root/'short'/'risk.npy')
    mask6=np.zeros(len(target),bool)
    for pos in target.groupby('query_id',sort=False).indices.values():
        mask6[pos[stable_order(target.id.iloc[pos],base[pos])[:min(SHORT_K,len(pos))]]]=True
    assert np.all(mask[mask6]) and np.isfinite(weights).all()
    assert len(base)==len(target) and np.isfinite(base).all() and np.isfinite(short[mask]).all()
    channels=dict(full_first=first,full_tta=base,
        cascade=cascade_scores(target,base,short,mask,weights),
        cascade_fixed45=cascade_scores(target,base,short,mask,SHORT_WEIGHT),
        cascade_top6_fixed45=cascade_scores(target,base,short,mask6,SHORT_WEIGHT),
        shortlist_only=cascade_scores(target,base,short,mask,1.0),
        cascade_strong=cascade_scores(target,base,short,mask,.70),
        cascade_light=cascade_scores(target,base,short,mask,.20),
        cascade_flat35=cascade_scores(target,base,short,mask,.35),
        cascade_flat55=cascade_scores(target,base,short,mask,.55),
        cascade_guarded=cascade_scores(target,base,short,mask,guarded_weights(target,base,short,mask,weights)))

    write_submission(raw_test,channels['cascade'][n_val:],root/'submission_v11_baseline.csv')
    write_submission(raw_test,channels['cascade'][n_val:],Path('submission.csv'))
    write_submission(raw_test,channels['cascade'][n_val:],root/'submission.csv')
    extra_path=root/'full'/'extra.npy'
    extra=np.load(extra_path) if extra_path.is_file() else np.full(len(target),np.nan)
    reverse=np.load(root/'full'/'reverse.npy'); sf=np.load(root/'short'/'first.npy'); sr=np.load(root/'short'/'reverse.npy')

    try:
        channels.update(strong_channel_bank(target,base,short,mask,weights,first,reverse,sf,sr,extra))
        if held is not None:
            features=calibration_features(target,base,short,mask,weights,first,reverse,sf,sr,extra)
            calibration=select_v13(held,target,channels,features,components,n_val,root)
        else:
            channels['final']=channels['cascade'].copy(); calibration={'selected':'cascade','reason':'full mode: no held labels'}
            atomic_json(root/'calibration.json',calibration)
    except Exception as exc:
        import traceback; traceback.print_exc()
        channels['final']=channels['cascade'].copy(); calibration={'selected':'cascade','reason':'optional calibration failed','error':repr(exc)}
        atomic_json(root/'calibration.json',calibration)
    for k,s in channels.items():
        write_submission(raw_test,s[n_val:],root/f'submission_{k}.csv')
    write_submission(raw_test,channels['final'][n_val:],Path('submission.csv'))
    write_submission(raw_test,channels['final'][n_val:],root/'submission.csv')
    doc=target[['id','query_id','document_id','article_id','query','title','text']].copy()
    doc['shortlist']=mask; doc['short_score']=short
    chosen_weights=weights if calibration['selected']=='cascade' else np.full(len(target),{'shortlist_only':1.,'cascade_top6_fixed45':.45}.get(calibration['selected'],np.nan))
    doc['risk']=risk; doc['adaptive_weight']=weights; doc['specialist_weight']=chosen_weights
    doc['full_extra_raw']=extra
    doc['full_forward_raw']=first; doc['full_reverse_raw']=np.load(root/'full'/'reverse.npy')
    doc['short_forward_raw']=np.load(root/'short'/'first.npy')
    doc['short_reverse_raw']=np.load(root/'short'/'reverse.npy')
    for k,s in channels.items():
        doc[k+'_score']=s; doc[k+'_rank']=rank_vector(target,s)
    doc.iloc[n_val:].to_csv(root/'test_document_scores.csv',index=False)
    if held is not None:
        val=doc.iloc[:n_val].copy(); val['relevance']=held.relevance.to_numpy()
        val.to_csv(root/'validation_document_errors.csv',index=False)
        validation_report(held,channels,root,components,n_val)
        detailed_diagnostics(held,channels,base,short,mask,chosen_weights,risk,n_val,root)
    atomic_json(root/'status.json',dict(complete=True,mode=mode,elapsed_minutes=(time.monotonic()-start)/60,
                    branches=statuses,extra_coverage=read_json(root/'full'/'extra_coverage.json'),budget_minutes=TOTAL_MINUTES,short_k=[SHORT_K,SHORT_K_HARD],short_weight='0.25+0.35*risk',selected=calibration['selected'],train_queries=fit.query_id.nunique()))
    try: compact_export(root)
    except Exception as exc: print('COMPACT_EXPORT_FAILED',repr(exc),flush=True)
    print('V13 COMPLETE; elapsed minutes',(time.monotonic()-start)/60,flush=True)


def recover_current(root,raw_test,target,held,n_val,start,mode,statuses):
    failures=[]
    for name in ('submission.csv','submission_cascade_required.csv','submission_full_tta.csv','submission_full_first.csv'):
        source=root/name
        if not source.is_file(): continue
        try:
            if name.startswith('submission_full') and statuses.get('full',{}).get('optimizer_updates',0)<1:

                if read_json(root/'full'/'training.json').get('optimizer_updates',0)<1: continue
            if name=='submission_cascade_required.csv' and any(read_json(root/b/'training.json').get('optimizer_updates',0)<1 for b in BRANCHES): continue
            data=pd.read_csv(source,dtype={'id':str})
            if list(data.columns)!=['id','prediction'] or not data.id.is_unique or set(data.id)!=set(raw_test.id.astype(str)) or not np.isfinite(data.prediction).all():
                raise ValueError('Invalid recovered submission')
            aligned=raw_test.id.astype(str).map(data.set_index('id').prediction).to_numpy()
            write_submission(raw_test,aligned,Path('submission.csv'))
            if name!='submission.csv': write_submission(raw_test,aligned,root/'submission.csv')
            state=dict(complete=True,degraded=True,selected=name,mode=mode,elapsed_minutes=(time.monotonic()-start)/60,branches=statuses,
                       warning='Recovered complete predictions from THIS run; full cascade or diagnostics were not completed. No old-model substitution.')
            atomic_json(root/'status.json',state)
            if held is not None and name in ('submission_full_tta.csv','submission_full_first.csv'):
                fname='scores.npy' if name.endswith('tta.csv') else 'first.npy'
                if (root/'full'/fname).exists():
                    scores=np.load(root/'full'/fname)
                    if len(scores)==len(target) and np.isfinite(scores[:n_val]).all():
                        atomic_json(root/'metrics_recovered.json',{'full_ndcg':float(ndcg_vector(held,scores[:n_val]).mean()),'selected':name})
            try: compact_export(root)
            except Exception as exc: print('COMPACT_EXPORT_FAILED',repr(exc),flush=True)
            print('RECOVERED CURRENT RUN:',json.dumps(state,ensure_ascii=False),flush=True)
            return
        except (OSError,ValueError,KeyError,AssertionError) as exc:
            failures.append(f'{name}: {exc}')
    atomic_json(root/'status.json',dict(complete=False,degraded=True,mode=mode,branches=statuses,errors=failures,warning='No complete prediction vector from this run'))
    raise RuntimeError('No valid complete predictions from this run. Inspect full.log and short.log. '+str(failures))


def finalize_saved(root):

    payload=pickle.load(open(root/'payload.pkl','rb'))
    statuses={b:read_json(root/b/'status.json') for b in BRANCHES}
    raw_test=payload.get('raw_test'); held=payload.get('held'); components=payload.get('components')
    if raw_test is None:
        raw_train,raw_test=read_data(find_data_files()); train,test=clean_data(raw_train,raw_test)
        if payload['n_val']:
            fi,vi,components=connected_query_split(train); held=train.iloc[vi].reset_index(drop=True)
            assert held.id.tolist()==payload['target'].id.iloc[:payload['n_val']].tolist()
            assert train.iloc[fi].id.tolist()==payload['fit'].id.tolist()
        assert test.id.tolist()==payload['target'].id.iloc[payload['n_val']:].tolist()
    if all(v.get('complete') and v.get('optimizer_updates',0)>0 for v in statuses.values()):
        try:
            finalize(root,raw_test,payload['target'],held,components,payload['n_val'],payload['start'],payload['mode'],statuses,payload['fit'])
            return
        except Exception:
            import traceback; traceback.print_exc()
    recover_current(root,raw_test,payload['target'],held,payload['n_val'],payload['start'],payload['mode'],statuses)


if __name__=='__main__':
    parser=argparse.ArgumentParser(); parser.add_argument('--worker',choices=list(BRANCHES)); parser.add_argument('--root'); parser.add_argument('--finalize-only',action='store_true')
    args=parser.parse_args()
    if args.worker: worker(args.worker,Path(args.root))
    elif args.finalize_only: finalize_saved(Path(args.root))
    else: main()

## Запуск обучения и прогнозирования

In [4]:

import signal, time, traceback
script=Path('_jina_v13_extended.py').resolve()
log_path=Path('jina_v13_extended.log')
root=Path('amml_v13_extended')/RUN_MODE
hard_end=float(os.environ['V13_FINAL_START'])+int(os.environ['V13_TOTAL_MINUTES'])*60
end=hard_end-100
if not script.is_file():
    raise FileNotFoundError(script)
if time.monotonic()>=end:
    raise TimeoutError('Extended budget already expired; restart from the first cell or recover completed outputs.')

def show_worker_errors():
    for p in [log_path,root/'full.log',root/'short.log']:
        if p.is_file():
            print(f'\n===== {p} (last lines) =====', flush=True)
            print(p.read_text(encoding='utf-8',errors='replace').replace('\r','\n')[-14000:],flush=True)
    for p in [Path('submission.csv'), root/'submission_cascade_required.csv', root/'submission_full_tta.csv']:
        if p.is_file(): print('Saved file:',p,flush=True)

with log_path.open('w',encoding='utf-8') as log:
    proc=subprocess.Popen([sys.executable,'-u',str(script)],stdout=log,stderr=subprocess.STDOUT,start_new_session=True)
    try:
        with log_path.open('r',encoding='utf-8',errors='replace') as reader:
            while proc.poll() is None:
                reader.read()
                if time.monotonic()>=end:
                    raise TimeoutError('Extended outer deadline; inspect saved outputs')
                time.sleep(min(5,max(0,end-time.monotonic())))
            reader.read()
        if proc.returncode:
            raise RuntimeError(f'V13 SAFE worker exit={proc.returncode}')
    except BaseException:
        if proc.poll() is None:
            try: os.killpg(proc.pid,signal.SIGTERM)
            except ProcessLookupError: pass
            try: proc.wait(timeout=5)
            except subprocess.TimeoutExpired:
                try: os.killpg(proc.pid,signal.SIGKILL)
                except ProcessLookupError: pass
                proc.wait()
        log.flush()
        show_worker_errors()

        remaining=hard_end-15-time.monotonic()
        import json
        try:
            current=json.loads((root/'run_identity.json').read_text())['start']==float(os.environ['V13_FINAL_START'])
        except (OSError,ValueError,KeyError): current=False
        if isinstance(sys.exc_info()[1],Exception) and current and remaining>10 and (root/'payload.pkl').exists():
            recovery=subprocess.run([sys.executable,'-u',str(script),'--finalize-only','--root',str(root.resolve())],timeout=min(remaining-5,90))
            if recovery.returncode!=0: raise
        else: raise


## Итоговые результаты

In [5]:
import json
from IPython.display import display, FileLink

root = Path('amml_v13_extended') / RUN_MODE
status = json.loads((root / 'status.json').read_text())
print('Статус:', 'Завершено' if status['complete'] else 'Не завершено')
print(f"Время выполнения, мин: {status['elapsed_minutes']:.2f}")
print('Итоговый вариант:', status.get('selected', '—'))
if status.get('degraded'):
    print(status.get('warning', 'Восстановлены доступные предсказания.'))
if (root / 'metrics.json').is_file():
    metrics = json.loads((root / 'metrics.json').read_text())
    for key, label in [
        ('full_first', 'Full Jina'),
        ('full_tta', 'Full Jina + TTA'),
        ('cascade', 'Каскад'),
        ('final', 'NDCG@10 (итоговый, описательный)'),
        ('pooled_oof_ndcg', 'NDCG@10 (OOF + verification)'),
        ('final_top1', 'Top-1 (итоговый)'),
    ]:
        if metrics.get(key) is not None:
            print(f'{label}: {metrics[key]:.6f}')
    if metrics.get('verification_ndcg') is not None:
        print(f"NDCG@10 (verification): {metrics['verification_ndcg']:.6f}")
    print('Полные эпохи:', all(b.get('full_epoch', False) for b in status['branches'].values()))
elif (root / 'metrics_recovered.json').is_file():
    print(json.loads((root / 'metrics_recovered.json').read_text()))
if Path('submission.csv').is_file():
    display(FileLink('submission.csv'))

Статус: Завершено
Время выполнения, мин: 222.91
Итоговый вариант: residual_calibrated
Full Jina: 0.964814
Full Jina + TTA: 0.965287
Каскад: 0.967805
NDCG@10 (итоговый, описательный): 0.968447
NDCG@10 (OOF + verification): 0.968882
Top-1 (итоговый): 0.957774
NDCG@10 (verification): 0.969321
Полные эпохи: True
